# Spark API - Global Route Cost Comparison

This script compares route costs between global load and discharge ports. 

For a full explanation of how to import our Hubs Cargo or Spark30S data, please refer to our Python Jupyter Notebook Code Samples:

https://www.sparkcommodities.com/api/code-examples/jupyter.html


### Have any questions?

If you have any questions regarding our API, or need help accessing specific datasets, please contact us at:

__data@sparkcommodities.com__

or refer to our API website for more information about this endpoint:
https://www.sparkcommodities.com/api/lng-freight/routes.html

__N.B. This script requires a Freight subscription__


## 1. Importing Data

In [ ]:
import json
import os
import sys
import pandas as pd
import numpy as np
from base64 import b64encode
from urllib.parse import urljoin
from io import StringIO
import datetime

try:
    from urllib import request, parse
    from urllib.error import HTTPError
except ImportError:
    raise RuntimeError("Python 3 required")

In [ ]:
API_BASE_URL = "https://api.sparkcommodities.com"


def retrieve_credentials(file_path=None):
    """
    Find credentials either by reading the client_credentials file or reading
    environment variables
    """
    if file_path is None:
        client_id = os.getenv("SPARK_CLIENT_ID")
        client_secret = os.getenv("SPARK_CLIENT_SECRET")
        if not client_id or not client_secret:
            raise RuntimeError(
                "SPARK_CLIENT_ID and SPARK_CLIENT_SECRET environment vars required"
            )
    else:
        # Parse the file
        if not os.path.isfile(file_path):
            raise RuntimeError("The file {} doesn't exist".format(file_path))

        with open(file_path) as fp:
            lines = [l.replace("\n", "") for l in fp.readlines()]

        if lines[0] in ("clientId,clientSecret", "client_id,client_secret"):
            client_id, client_secret = lines[1].split(",")
        else:
            print("First line read: '{}'".format(lines[0]))
            raise RuntimeError(
                "The specified file {} doesn't look like to be a Spark API client "
                "credentials file".format(file_path)
            )

    print(">>>> Found credentials!")
    
    return client_id, client_secret


def do_api_post_query(uri, body, headers):
    """
    OAuth2 authentication requires a POST request with client credentials before accessing the API.
    This POST request will return an Access Token which will be used for the API GET request.
    """
    url = urljoin(API_BASE_URL, uri)

    data = json.dumps(body).encode("utf-8")

    # HTTP POST request
    req = request.Request(url, data=data, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 201. Raise an error if this is not the case
    assert response.status == 201, resp_content

    # The server returned a JSON response
    content = json.loads(resp_content)

    return content


def do_api_get_query(uri, access_token, format='json'):
    """
    After receiving an Access Token, we can request information from the API.
    Supports both JSON (default) and CSV responses.
    """
    url = urljoin(API_BASE_URL, uri)

    if format == 'json':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "application/json",
        }
    elif format == 'csv':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "text/csv",
        }
    else:
        raise ValueError("The format parameter only takes 'csv' or 'json' as inputs")

    print("Fetching {}".format(url))

    # HTTP GET request
    req = request.Request(url, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 200. Raise an error if this is not the case
    assert response.status == 200, resp_content

    # Storing response based on requested format
    if format == 'json':
        content = json.loads(resp_content)
    elif format == 'csv':
        content = resp_content

    return content


def get_access_token(client_id, client_secret):

    payload = "{}:{}".format(client_id, client_secret).encode()
    headers = {
        "Authorization": b64encode(payload).decode(),
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    body = {
        "grantType": "clientCredentials",
    }

    content = do_api_post_query(uri="/oauth/token/", body=body, headers=headers)

    print(
        ">>>> Successfully fetched an access token {}****, valid {} seconds.".format(
            content["accessToken"][:5], content["expiresIn"]
        )
    )

    return content["accessToken"]

## N.B. Credentials

Here we call the above functions, and input the file path to our credentials.

N.B. You must have downloaded your client credentials CSV file before proceeding. Please refer to the API documentation if you have not dowloaded them already.

In [ ]:
# Insert file path to your client credentials here
client_id, client_secret = retrieve_credentials(file_path="/tmp/client_credentials.csv")

# Authenticate:
access_token = get_access_token(client_id, client_secret)

# 2. Fetching Routes Reference Data

Fetches information on all available routes and their associated tickers

In [ ]:
## Define function for listing routes reference data

def list_routes_refdata(access_token):

    content = do_api_get_query(uri="/v1.0/routes/", access_token=access_token)

    reldates = content["data"]["sparkReleaseDates"]

    data_dict = content["data"]

    return reldates, data_dict


# Fetch all reference data
reldates, refdata_dict = list_routes_refdata(access_token)

# formatting as a Dataframe
available_df = pd.json_normalize(refdata_dict['routes'])
available_df = available_df.rename(columns={
    'uuid' : 'UUID',
    'via' : 'Via',
    'loadPort.uuid' : 'LoadPortUUID',
    'loadPort.type' : 'LoadPortType',
    'loadPort.region' : 'LoadPortRegion',
    'loadPort.name' : 'LoadPortName',
    'dischargePort.uuid' : 'DischargePortUUID',
    'dischargePort.type' : 'DischargePortType',
    'dischargePort.region' : 'DischargePortRegion',
    'dischargePort.name' : 'DischargePortName',
})

available_df.head(3)

## 3. Definiting Routes Data Fetch Function

In [ ]:
## Defining

def fetch_route_data(access_token, ticker, release, congestion=None):
    """
    For a route, fetch then display the route details

    # Procedure:

    Do GET queries to https://api.sparkcommodities.com/v1.0/routes/{route_uuid}/
    with a Bearer token authorization HTTP header.
    """

    query_params = "?release-date={}".format(release)
    if congestion is not None:
        query_params += "&congestion-days={}".format(congestion)

    uri = "/v1.0/routes/{}/{}".format(ticker, query_params)
    #print(uri)

    content = do_api_get_query(
        uri="/v1.0/routes/{}/{}".format(ticker, query_params),
        access_token=access_token,
    )

    my_dict = content["data"]

    return my_dict


# 4. Load & Discharge Port Inputs

Input the list of Load and Discharge ports you'd like to include in the table. Cross reference with the Routes Reference table above to make sure the route combinations you input are assessed by Spark.

__Looking for a a Route Cost we don't currently provide?__ Contact us at info@sparkcommodities.com to request any new routes.

In [ ]:
# Defining which ports to use in our table
my_loads = ['Sabine Pass', 'Atlantic LNG', 'Bethioua', 'Hammerfest', 'Bonny LNG','Yamal',  'Ras Laffan', 'Bintulu','Gorgon', 'GLNG']
my_discharges = ['Bahia','Gate','Swinoujscie','Fos Cavaou','Rovigo','Dahej','Rayong','Dapeng','Yung An','Tianjin','Incheon','Futtsu']


### Defining Function to pull data and save as Pandas DataFrame

The function takes 4 parameters:
- __loads:__ the list of Load Ports you'd like to use in your table
- __discharges:__ the list of Discharge Ports you'd like to use in your table
- __month:__ which month's prices you'd like to retrieve (0= spot, 1=M+1, etc.)
- __release:__ which published price release you'd like to use. Default is 'latest', which calls the latest route costs. Input a date (e.g. '2025-04-03') to call historical prices

In [ ]:
import numpy as np

def routes_table(loads,discharges, month, release='latest'):
    # initialising DataFrame
    table_df = pd.DataFrame(columns=discharges)
    table_df['FoB Ports'] = loads
    table_df = table_df.set_index('FoB Ports')

    # Iterating through load, discharge combinations and retrieving relevant data
    for l in loads:
        for d in discharges:
            a = available_df[(available_df['LoadPortName'] == l)\
            & (available_df['DischargePortName'] == d)]

            # retrieving the route UUID, making sure the via point is COGH where applicable
            try:
                vias = a['Via'].to_list()
                if len(a) == 1:
                    tick = a['UUID'].iloc[0]
                else: 
                    if 'cogh' in vias:
                        tick = a[a['Via'] == 'cogh']['UUID'].iloc[0]
                    else:
                        tick = a[(a['Via'] != 'panama') & \
                                 (a['Via'] != 'suez') & \
                                (a['Via'] != 'magellan-straits')]['UUID'].iloc[0]
                    
                # calling route data with UUID
                if release == 'latest':
                    my_dict = fetch_route_data(access_token, tick, release=reldates[0])
                else:
                    my_dict = fetch_route_data(access_token, tick, release=release)

                table_df.loc[l,d] = float(my_dict['dataPoints'][month]['costsInUsdPerMmbtu']['total'])
            
            except:
                table_df.loc[l,d] = np.nan
        
    return table_df

## Calling the Data

As an example, here we call the latest spot route costs for our chosen Load and Discharge ports.

In [ ]:
table_df = routes_table(my_loads,my_discharges, month=0, release = 'latest')
table_df

# Optional Formatting using 'plottable'

This section uses the Python package 'plottable' to apply style formatting to our Routes table. 'plottable' is not a standard Python package and needs to be downloaded separately - as such, the code below is commented out to avoid script breaks. Remove the {"""} at the beginning/end of the code to uncomment the code and run this section.

For more information on 'plottable', please refer to their documentation:

https://plottable.readthedocs.io/en/latest/index.html#

In [ ]:
"""

import matplotlib.pyplot as plt
from plottable import ColumnDefinition, Table
from plottable.cmap import normed_cmap
import matplotlib

# retrieving the columns ranked by standard deviation, and choosing one to use for the colourmap scaling
ci = list(table_df.std().sort_values().index)[-2]

# initiating figure
fig, ax = plt.subplots(figsize=(20, 8))


# defining columns of the table
col_defs = (
    [
        ColumnDefinition(
            name="FoB Ports",
            textprops={"ha": "left", "weight": "bold"},
            width=1.5,
            border='right',
        )
    ]
    + [
        ColumnDefinition(
            name=col,
            title=col,
            cmap=normed_cmap(table_df[ci], cmap=matplotlib.cm.coolwarm, num_stds=2.5),
            textprops={"ha": "center"},
            group="Discharge Locations",
        )
        for col in my_discharges
    ])

# creating table
table = Table(
    table_df,
    row_dividers=True,
    col_label_divider=True,
    column_definitions=col_defs,
    footer_divider=True,
    ax=ax,
    textprops={"fontsize": 12},
    row_divider_kw={"linewidth": 1, "linestyle": (0, (1, 5))},
    col_label_divider_kw={"linewidth": 1, "linestyle": "-"},
    column_border_kw={"linewidth": 1, "linestyle": "-"},
)

plt.show()

"""